# Orbital sunlight — the model

Readable copy of the physics. Formulas typeset in this notebook; each one also has a **plain** line.

If math looks like `\\frac`, open **`notebooks/the_model.html`** in a browser instead (double-click it).

This is the **model**, not the results. The study (tables, plots, verdict) is still ahead.

## What we are asking

A specular mirror in orbit reflects sunlight to Earth’s surface. For collecting area $A$ and altitude $h$:

1. How **bright** is the ground patch? ($I$, W/m²)
2. How **big** is it? ($D$, km)
3. How **long** does an overhead pass last? ($T$, min)

Which fact binds: **energy** (the Sun is a disk, so the spot cannot be squeezed arbitrarily small and bright) or the **orbital shutter** (the satellite races past)?

Geometry: dusk/dawn, looking straight down. Fold angle $\gamma = 45^\circ$. This is **not** midnight over a dark city.

## 1. Patch size

The Sun has a finite angular diameter $\alpha = 0.0093$ rad (full disk, about 0.53°). Flat or focusing, the ground image cannot be smaller than:

$$d = h, \qquad D = d\,\alpha, \qquad A_{\mathrm{image}} = \pi (D/2)^2$$

**Plain:** `D = h * alpha`  and  `A_image = pi * (D/2)**2`  (use metres).

Focusing does **not** shrink $D$.

## 2. Brightness

Collected power spread over the solar image. Not a lamp $1/r^2$.

$$I = \frac{E_0\,\eta\,A\cos\gamma\sin\varepsilon}{\pi (d\alpha/2)^2}$$

**Plain:** `I = E0 * eta * A * cos(gamma) * sin(epsilon) / A_image`

Snapshot: $\varepsilon = 90^\circ$ so `sin(epsilon) = 1`; $\gamma = 45^\circ$ so `cos(gamma) = 0.707`. Pass `eta` as **one** factor (`1` or `0.675`). Do not also multiply reflectance and atmosphere.

$E_0 = 1361$ W/m² (sunlight in space).

## 3. Pass duration

Orbit radius is Earth radius plus altitude, not altitude alone.

$$a = R_E + h, \qquad T = 2\pi\sqrt{a^3/\mu}$$

**Plain:** `a = R_earth + h`  and  `T_period = 2*pi*sqrt(a**3 / mu)`

Time above 30° elevation on an overhead pass:

$$\theta(\varepsilon) = \arccos((R_E/a)\cos\varepsilon) - \varepsilon, \qquad T_{\mathrm{useful}} = T\,\theta(\varepsilon_{\min})/\pi$$

**Plain:** `theta = arccos((R_earth/a)*cos(eps)) - eps`  (radians)  
`T_useful = T_period * theta / pi`  (seconds in code; `/60` for minutes).

Eclipse is not subtracted. Off-track passes are shorter.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.physics import solar_image, irradiance, pass_window, load_constants

h_m = 625_000
A = 55 * 55
img = solar_image(h_m)
I_ideal = irradiance(A, h_m, 1.0)
I_real = irradiance(A, h_m, 0.675)
pw = pass_window(h_m)

print(f"Patch diameter D = {img.D_m/1000:.3f} km")
print(f"Patch area      = {img.A_image_m2:.3e} m²")
print(f"I ideal (η=1)   = {I_ideal:.4f} W/m²")
print(f"I estimated     = {I_real:.4f} W/m²")
print(f"T useful        = {pw.T_useful_s/60:.2f} min  (overhead pass, geometric)")
print("(Full sun ~1000 W/m²; moonlight-class ~0.003 W/m² estimated)")

That cell is a **calibration point** (625 km, 55 m square), not the study. Next work is plots and a grid of $A$ and $h$, then comparison to lighting/PV thresholds.

Contract: `spec.md`. Calculator: `src/physics.py`. Question: `framing.md`.